# MegaDescriptor 파인튜닝 / 평가 — MPDD

- **데이터**: MPDD (Market-1501 스타일). `MPDD/pytorch/{train,val,gallery,query}/` 안에 `<id>_c<pose>s<seq>_<n>.jpg` 파일들.
- **평가**: MPDD 공식 프로토콜(query vs gallery) + 최종은 `processed_animals`(보호소 데이터).
- **CPU 주의**: 이 `.venv` 는 `torch+cpu`. 그러면 학습 셀은 자동 스킵되고 **zero-shot 평가만** 돈다.
  파인튜닝하려면 CUDA GPU 환경 + `torch` CUDA 빌드 필요.
- 위 → 아래 순서로 실행. 문제 생기면 커널 Restart 후 재실행.


In [21]:
import os, re, glob, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
import timm
import torchvision.transforms as T

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = DEVICE == "cuda"
PIN      = USE_CUDA
EXTS     = (".jpg", ".jpeg", ".png", ".webp")
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | DEVICE {DEVICE}")
if not USE_CUDA:
    print(">> CPU 환경: 학습 셀은 스킵되고 zero-shot 평가만 실행됩니다.")


torch 2.13.0+cpu | CUDA False | DEVICE cpu
>> CPU 환경: 학습 셀은 스킵되고 zero-shot 평가만 실행됩니다.


## 설정

In [14]:
def _find(cands):
    for p in cands:
        hit = glob.glob(p, recursive=True)
        if hit:
            return hit[0]
    return None

MPDD = _find([
    "dataset/Multi-pose dog dataset/MPDD/pytorch",
    "ML/dataset/Multi-pose dog dataset/MPDD/pytorch",
    "**/MPDD/pytorch",
])
SHELTER = _find(["processed_animals", "ML/processed_animals", "**/processed_animals"])
print("MPDD    :", MPDD)
print("SHELTER :", SHELTER)

CONFIG = {
    "model":        "hf-hub:BVRA/MegaDescriptor-B-224",  # B-224. L-384는 GPU 여유 있을 때
    "epochs":       20,
    "pk_p":         12,      # 배치 P개체
    "pk_k":         4,       # 개체당 K장 (배치 = P*K)
    "lr_head":      1e-4,
    "lr_backbone":  1e-5,
    "wd":           1e-4,
    "freeze_until": 2,       # 학습시킬 첫 Swin stage(0~3). -1 전체, 99 헤드만(linear probe)
    "seed":         0,
    "out":          "megadescriptor_ft.pt",
    "allow_cpu_train": False,  # True로 하면 CPU에서도 학습 강행 (매우 느림)
}
C = CONFIG
random.seed(C["seed"]); np.random.seed(C["seed"]); torch.manual_seed(C["seed"])
RUN_TRAIN = USE_CUDA or C["allow_cpu_train"]


MPDD    : dataset/Multi-pose dog dataset/MPDD/pytorch
SHELTER : processed_animals


## 데이터 로더 (Market-1501 스타일)

In [15]:
_MKT = re.compile(r"^(\d+)_c(\d+)s\d+")

def df_from_market(folder):
    """<id>_c<pose>s<seq>_<n>.jpg  ->  identity / cam / path(파일명)"""
    rows = []
    for f in sorted(os.listdir(folder)):
        m = _MKT.match(f)
        if f.lower().endswith(EXTS) and m:
            rows.append({"identity": m.group(1), "cam": int(m.group(2)), "path": f})
    return pd.DataFrame(rows)

def df_from_folder(root):
    """<root>/<개체ID>/*.jpg  ->  identity / path(개체ID/파일명).  (processed_animals 용)"""
    rows = []
    for d in sorted(os.listdir(root)):
        dd = os.path.join(root, d)
        if not os.path.isdir(dd):
            continue
        for f in sorted(os.listdir(dd)):
            if f.lower().endswith(EXTS):
                rows.append({"identity": d, "cam": -1, "path": os.path.join(d, f)})
    return pd.DataFrame(rows)


tr  = df_from_market(f"{MPDD}/train")
va  = df_from_market(f"{MPDD}/val")
gal = df_from_market(f"{MPDD}/gallery")
qry = df_from_market(f"{MPDD}/query")

id2lab = {c: i for i, c in enumerate(sorted(tr["identity"].unique()))}
tr = tr.copy(); tr["label"] = tr["identity"].map(id2lab)
va = va[va["identity"].isin(id2lab)].copy(); va["label"] = va["identity"].map(id2lab).astype(int)
N_CLASSES = len(id2lab)

print(f"train   {len(tr):4d}장 / {N_CLASSES}개체")
print(f"val     {len(va):4d}장 (train과 동일 개체, 모니터용)")
print(f"gallery {len(gal):4d}장 / {gal['identity'].nunique()}개체   query {len(qry)}장 / {qry['identity'].nunique()}개체")
print("train ∩ gallery 개체:", len(set(tr['identity']) & set(gal['identity'])), "(0이어야 정상 = open-set)")


train    921장 / 95개체
val      111장 (train과 동일 개체, 모니터용)
gallery  521장 / 96개체   query 103장 / 95개체
train ∩ gallery 개체: 0 (0이어야 정상 = open-set)


## 모델 + transform

In [16]:
model = timm.create_model(C["model"], pretrained=True, num_classes=0).to(DEVICE)
cfg = timm.data.resolve_model_data_config(model)
SIZE = cfg["input_size"][-1]
print("input", SIZE, "| embed dim", model.num_features)

train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(),
    T.Normalize(cfg["mean"], cfg["std"]),
])
eval_tf = timm.data.create_transform(**cfg, is_training=False)


input 224 | embed dim 1024


## Dataset · P×K 샘플러 · ArcFace 헤드

In [17]:
class ReIDDataset(Dataset):
    def __init__(self, df, root, tf):
        self.paths = df["path"].tolist()
        self.y = df["label"].to_numpy() if "label" in df else np.zeros(len(df), int)
        self.root, self.tf = root, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.paths[i])).convert("RGB")
        return self.tf(img), int(self.y[i])


class PKSampler(Sampler):
    """배치 = P개체 x K장. metric learning 필수."""
    def __init__(self, labels, p, k, steps=None):
        self.labels = np.asarray(labels)
        self.p, self.k = p, k
        self.by_id = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        self.ids = list(self.by_id)
        self.steps = steps or max(1, len(self.labels) // (p * k))
    def __iter__(self):
        for _ in range(self.steps):
            picks = np.random.choice(self.ids, self.p, replace=len(self.ids) < self.p)
            batch = []
            for c in picks:
                idx = self.by_id[c]
                batch += list(np.random.choice(idx, self.k, replace=len(idx) < self.k))
            yield batch
    def __len__(self):
        return self.steps


class ArcFaceHead(nn.Module):
    def __init__(self, in_dim, n_classes, s=64.0, m=0.5):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes, in_dim))
        nn.init.xavier_uniform_(self.W)
        self.s = s
        self.cos_m, self.sin_m = np.cos(m), np.sin(m)
        self.th, self.mm = np.cos(np.pi - m), np.sin(np.pi - m) * m
    def forward(self, feats, labels):
        cos = F.linear(F.normalize(feats), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt(1.0 - cos ** 2)
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        oh = F.one_hot(labels, cos.size(1)).float()
        return (oh * phi + (1.0 - oh) * cos) * self.s


## 평가 함수

In [18]:
def _ap_cmc(rel):
    if not rel.any():
        return 0.0, 0, 0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / rel.sum()), int(rel[0]), int(rel[:5].any())


@torch.no_grad()
def extract(model, loader):
    model.eval()
    E, L = [], []
    for x, y in loader:
        v = F.normalize(model(x.to(DEVICE)))
        E.append(v.cpu().numpy()); L.append(y.numpy())
    return np.concatenate(E), np.concatenate(L)


def evaluate_qg(q_emb, q_lab, g_emb, g_lab, q_cam, g_cam, exclude_same_cam=True):
    """MPDD query/gallery. Market 규칙: 같은 id·같은 pose 는 junk 로 제외."""
    q_lab, g_lab = np.asarray(q_lab), np.asarray(g_lab)
    q_cam, g_cam = np.asarray(q_cam), np.asarray(g_cam)
    sim = q_emb @ g_emb.T
    aps = c1 = c5 = 0.0
    for i in range(len(q_lab)):
        order = np.argsort(sim[i])[::-1]
        if exclude_same_cam:
            junk = (g_lab == q_lab[i]) & (g_cam == q_cam[i])
            order = order[~junk[order]]
        ap, h1, h5 = _ap_cmc(g_lab[order] == q_lab[i])
        aps += ap; c1 += h1; c5 += h5
    n = len(q_lab)
    return aps / n, c1 / n, c5 / n


def evaluate_map_split(emb, labels, gallery_frac=0.5, seed=0):
    """개체당 절반 gallery / 절반 query. processed_animals 용."""
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    g_idx, q_idx = [], []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0].copy()
        rng.shuffle(idx)
        if len(idx) <= 1:
            g_idx += idx.tolist(); continue
        k = max(1, round(len(idx) * gallery_frac))
        g_idx += idx[:k].tolist(); q_idx += idx[k:].tolist()
    g_idx, q_idx = np.array(g_idx), np.array(q_idx)
    g_emb, g_lab = emb[g_idx], labels[g_idx]
    sim = emb[q_idx] @ g_emb.T
    aps = c1 = c5 = 0.0
    for row, gt in zip(sim, labels[q_idx]):
        ap, h1, h5 = _ap_cmc(g_lab[np.argsort(row)[::-1]] == gt)
        aps += ap; c1 += h1; c5 += h5
    q = len(q_idx)
    return aps / q, c1 / q, c5 / q


def evaluate_map_split_mean(emb, labels, seeds=range(5)):
    r = np.array([evaluate_map_split(emb, labels, seed=s) for s in seeds])
    return r.mean(0), r.std(0)          # ([mAP, Top-1, Top-5] 평균, 표준편차)


def shelter_report(model, tag):
    if not SHELTER:
        print("processed_animals 없음 - 스킵"); return
    d = df_from_folder(SHELTER)
    d["label"] = d["identity"].astype("category").cat.codes
    ld = DataLoader(ReIDDataset(d, SHELTER, eval_tf), batch_size=64, num_workers=0, pin_memory=PIN)
    e, l = extract(model, ld)
    (mean, std) = evaluate_map_split_mean(e, l)
    print(f"[{tag} / processed_animals] 개체 {d['identity'].nunique()} / 이미지 {len(d)}")
    print(f"   split  mAP {mean[0]:.4f}±{std[0]:.4f} | Top-1 {mean[1]:.4f}±{std[1]:.4f} | Top-5 {mean[2]:.4f}±{std[2]:.4f}")


## 학습 준비 (freeze · 헤드 · 로더 · 옵티마이저)

In [19]:
def set_trainable(model, freeze_until):
    for p in model.parameters():
        p.requires_grad = True
    if freeze_until < 0:
        return
    for p in model.patch_embed.parameters():
        p.requires_grad = False
    stages = list(model.layers) if hasattr(model, "layers") else []
    for i, st in enumerate(stages):
        for p in st.parameters():
            p.requires_grad = i >= freeze_until


set_trainable(model, C["freeze_until"])
head = ArcFaceHead(model.num_features, N_CLASSES).to(DEVICE)
n_bb = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 backbone 파라미터 {n_bb/1e6:.1f}M / 전체 {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

_mk = lambda df, sub: DataLoader(ReIDDataset(df, f"{MPDD}/{sub}", eval_tf),
                                 batch_size=64, num_workers=0, pin_memory=PIN)
tr_ld  = DataLoader(ReIDDataset(tr, f"{MPDD}/train", train_tf),
                    batch_sampler=PKSampler(tr["label"].to_numpy(), C["pk_p"], C["pk_k"]),
                    num_workers=0, pin_memory=PIN)
gal_ld = _mk(gal, "gallery")
qry_ld = _mk(qry, "query")

bb = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{"params": bb, "lr": C["lr_backbone"]},
                         {"params": head.parameters(), "lr": C["lr_head"]}],
                        weight_decay=C["wd"])
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=C["epochs"])
scaler = torch.amp.GradScaler(enabled=USE_CUDA)


학습 backbone 파라미터 84.6M / 전체 86.7M


## Zero-shot 기준선 (학습 전, 항상 실행)

In [20]:
q0e, q0l = extract(model, qry_ld)
g0e, g0l = extract(model, gal_ld)
m, t1, t5 = evaluate_qg(q0e, q0l, g0e, g0l, qry["cam"].to_numpy(), gal["cam"].to_numpy())
print(f"[zero-shot / MPDD q-g]  mAP {m:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")
shelter_report(model, "zero-shot")


[zero-shot / MPDD q-g]  mAP 1.0000  Top-1 1.0000  Top-5 1.0000
[zero-shot / processed_animals] 개체 344 / 이미지 1358
   split  mAP 0.5538±0.0053 | Top-1 0.6516±0.0081 | Top-5 0.8075±0.0090


## 학습 (GPU에서만. CPU면 자동 스킵)

In [ ]:
if not RUN_TRAIN:
    print("학습 스킵: CUDA 없음. (CONFIG['allow_cpu_train']=True 로 강행 가능하나 매우 느림)")
else:
    best = -1.0
    for ep in range(1, C["epochs"] + 1):
        model.train(); head.train()
        losses = []
        for x, y in tr_ld:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast(device_type=DEVICE, enabled=USE_CUDA):
                loss = F.cross_entropy(head(model(x), y), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sched.step()

        qe, ql = extract(model, qry_ld)
        ge, gl = extract(model, gal_ld)
        mAP, t1, t5 = evaluate_qg(qe, ql, ge, gl, qry["cam"].to_numpy(), gal["cam"].to_numpy())
        print(f"[{ep:02d}/{C['epochs']}] loss {np.mean(losses):.3f} | "
              f"MPDD q-g  mAP {mAP:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")
        if mAP > best:
            best = mAP
            torch.save({"model": C["model"], "backbone": model.state_dict(),
                        "id2lab": id2lab, "cfg": cfg, "mpdd_mAP": best}, C["out"])
            print(f"    -> saved {C['out']}  (mAP {best:.4f})")
    print(f"\nbest MPDD q-g mAP {best:.4f}")


## 최종 평가 — 파인튜닝 backbone 을 보호소 데이터에

In [ ]:
if os.path.exists(C["out"]):
    ck = torch.load(C["out"], map_location=DEVICE)
    model.load_state_dict(ck["backbone"]); model.eval()
    qe, ql = extract(model, qry_ld); ge, gl = extract(model, gal_ld)
    m, t1, t5 = evaluate_qg(qe, ql, ge, gl, qry["cam"].to_numpy(), gal["cam"].to_numpy())
    print(f"[finetuned / MPDD q-g]  mAP {m:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")
    shelter_report(model, "finetuned")
else:
    print("체크포인트 없음 (학습 안 함). zero-shot 셀 결과를 기준으로 사용.")
